# Homework 06: Data Preprocessing

This notebook loads the raw SPY exercise data, applies reusable cleaning functions, saves the processed result, and compares the dataset before and after cleaning.

In [1]:
from pathlib import Path
import sys

import pandas as pd

In [2]:
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name != 'homework06':
    candidate = PROJECT_DIR / 'homework' / 'homework06'
    if candidate.is_dir():
        PROJECT_DIR = candidate

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.cleaning import drop_missing, fill_missing_median, normalize_data

RAW_PATH = PROJECT_DIR / 'data' / 'raw' / 'spy_preprocessing_raw.csv'
PROCESSED_PATH = PROJECT_DIR / 'data' / 'processed' / 'spy_preprocessing_cleaned.csv'

## 1. Load and inspect raw data

The source is the Homework 04 SPY dataset with four deliberately blanked cells for this cleaning exercise. This is documented so the exercise artifacts are not mistaken for genuine market-data gaps.

In [3]:
original_df = pd.read_csv(RAW_PATH, parse_dates=['Date'])
print(f'Original shape: {original_df.shape}')
pd.DataFrame({
    'dtype': original_df.dtypes.astype(str),
    'missing': original_df.isna().sum(),
})

Original shape: (252, 7)


,dtype,missing
Date,"datetime64[ns, UTC]",1
Adj Close,float64,0
Close,float64,1
High,float64,0
Low,float64,0
Open,float64,1
Volume,float64,1


## 2. Apply modular cleaning functions

Assumptions:

1. `Close` and `Open` are continuous price measurements, so their missing values can be median-imputed.
2. `Date` is the observation identifier and `Volume` is a critical activity measure; rows missing either field are dropped instead of guessed.
3. Price columns are min-max scaled for comparability. Volume stays in shares to preserve its business meaning.

In [4]:
price_columns = ['Adj Close', 'Close', 'High', 'Low', 'Open']

filled_df = fill_missing_median(original_df, columns=['Close', 'Open'])
complete_df = drop_missing(filled_df, subset=['Date', 'Volume'])
cleaned_df = normalize_data(complete_df, columns=price_columns)

cleaned_df.head()

,Date,Adj Close,Close,High,Low,Open,Volume
0,2025-08-19 00:00:00+00:00,0.016622,0.372901,0.031852,0.062949,0.039847,69750700.0
1,2025-08-20 00:00:00+00:00,0.005225,0.042081,0.000000,0.025111,0.342031,88890300.0
4,2025-08-25 00:00:00+00:00,0.034454,0.071962,0.040298,0.089428,0.046370,51274300.0
5,2025-08-26 00:00:00+00:00,0.052488,0.090398,0.041873,0.084092,0.033324,51581600.0
6,2025-08-27 00:00:00+00:00,0.062342,0.100473,0.055186,0.103592,0.050128,48341100.0


## 3. Validate and save the processed dataset

In [5]:
checks = {
    'expected_rows_removed': len(cleaned_df) == len(original_df) - 2,
    'no_missing_values': not cleaned_df.isna().any().any(),
    'price_min_is_zero': cleaned_df[price_columns].min().eq(0.0).all(),
    'price_max_is_one': cleaned_df[price_columns].max().eq(1.0).all(),
    'source_not_mutated': original_df.isna().sum().sum() == 4,
}
checks['all_checks_pass'] = all(checks.values())
assert checks['all_checks_pass'], checks
pd.Series(checks, name='passed').to_frame()

,passed
expected_rows_removed,True
no_missing_values,True
price_min_is_zero,True
price_max_is_one,True
source_not_mutated,True
all_checks_pass,True


In [6]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv(PROCESSED_PATH, index=False)
reloaded_df = pd.read_csv(PROCESSED_PATH, parse_dates=['Date'])
assert reloaded_df.shape == cleaned_df.shape
print(f'Saved {reloaded_df.shape[0]} rows and {reloaded_df.shape[1]} columns to {PROCESSED_PATH.relative_to(PROJECT_DIR)}')

Saved 250 rows and 7 columns to data/processed/spy_preprocessing_cleaned.csv


## 4. Original vs. cleaned comparison

In [7]:
comparison = pd.DataFrame({
    'original': [len(original_df), original_df.isna().sum().sum(), original_df['Close'].min(), original_df['Close'].max()],
    'cleaned': [len(cleaned_df), cleaned_df.isna().sum().sum(), cleaned_df['Close'].min(), cleaned_df['Close'].max()],
}, index=['rows', 'missing_cells', 'Close_min', 'Close_max'])
comparison

,original,cleaned
rows,252.000000,250.0
missing_cells,4.000000,0.0
Close_min,631.969971,0.0
Close_max,777.880005,1.0


## Reflection and tradeoffs

Median filling preserves two observations, but the substituted values understate uncertainty and slightly compress the distribution. Dropping two rows ensures every retained observation has a date and volume, but deletion can bias results when missingness is not random. Min-max scaling improves feature comparability and is easy to interpret, although future values outside the fitted historical range could fall outside `[0, 1]` and outliers determine the scale. In a production pipeline, scaling parameters should be fitted on training data only and saved for later reuse.